# Day-Ahead Electricity Demand Forecasting (+24h Horizon)

This notebook extends a baseline short-term forecasting pipeline to a **day-ahead electricity demand forecasting problem**.

The objective is to evaluate how predictive performance changes when:
- the forecasting horizon is shifted from +1h to +24h
- exogenous weather variables (temperature) are introduced
- statistical and machine learning models are compared under realistic constraints

## Workflow

1. Load and preprocess electricity + weather data  
2. Align and validate time-series consistency  
3. Construct lag features with +24h target shift  
4. Add weather-based exogenous variables  
5. Train baseline, statistical, and ML models  
6. Evaluate them with a chronological train/test split  
7. Validate robustness with time-series cross-validation 

## Pipeline overview

Raw data → Time Alignment → Feature Engineering → Target Shift (+24h) → Train/Test Split → Model Training → Evaluation

## Models compared

### Baselines
- Naive forecast (t-24)

### Statistical models
- Exponential Smoothing (Holt-Winters)

### Machine learning models
- Ridge Regression
- Random Forest Regressor
- Gradient Boosting Regressor

The goal is to move from a simplified forecasting setup to a more realistic **day-ahead electricity demand forecasting pipeline**, where predictive performance is evaluated under a +24h horizon and enriched with exogenous weather information.

Beyond accuracy, the focus is on model robustness, interpretability, and alignment with real-world energy forecasting constraints.

## Configuration

In [1]:
# =========================
# Project configuration
# =========================

DATA_PATH = "../data/PJME_hourly.csv"
TARGET_COL = "PJME_MW"

HORIZON = 24  # day-ahead forecasting

TEST_SIZE = 0.2
RANDOM_STATE = 42

# plotting window (first month)
PLOT_HOURS = 24 * 30

## Imports

In [2]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

from src.preprocessing import load_data, sort_by_time, run_data_checks
from src.features import create_features
from src.train import (
    make_Xy,
    make_train_test_split,
    build_ridge_pipeline,
    build_random_forest_model,
    build_gradient_boosting_model,
    fit_predict,
)
from src.evaluate import evaluate_regression

ImportError: cannot import name 'run_data_checks' from 'src.preprocessing' (/Users/antonio/Desktop/TRAINING/GITHUB/energy-demand-forecasting/src/preprocessing.py)

## Data loading and preprocessing

I load the electricity demand dataset, standardize the timestamp format, convert it to datetime, and sort the series chronologically to ensure proper time-series structure for forecasting.

In [4]:
df = load_data("../data/PJME_hourly.csv")
df = sort_by_time(df)

print("Raw shape:", df.shape)
df.head()

Raw shape: (145366, 1)


,PJME_MW
timestamp,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


## Data quality checks

Before feature engineering, I run a set of basic validation checks to ensure the time series is reliable and suitable for forecasting. These checks include:

- ordering and continuity of timestamps  
- detection of duplicates and missing values  
- verification of expected hourly frequency  
- assessment of missing time steps in the full time range   

These steps are critical because forecasting models rely on consistent temporal spacing for lag and rolling features.

In [5]:
checks, time_deltas = run_data_checks(df)

print("Data checks")
for name, value in checks.items():
    print(f"- {name}: {value}")

print("\nMost common time deltas:")
print(time_deltas)

NameError: name 'run_data_checks' is not defined

In [ ]:
# Remove duplicated timestamps, if any, keeping the first occurrence
df = df[~df.index.duplicated(keep="first")].copy()

print("Shape after duplicate handling:", df.shape)